# Metacritc data scraping


In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

In [2]:
# we will work with metacritic website to scrape data about pc games
# highest rated pc games of all time
# we will do the first page only for now
# then automate for all pages later

url ='https://www.metacritic.com/browse/game/pc/all/all-time/metascore/?releaseYearMin=1958&releaseYearMax=2025&platform=pc&page=1'

In [3]:
# we had to add headers to mimic a browser request
# because some websites block requests that seem to come from scripts or bots
# we added  user-agent and other common headers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Connection': 'keep-alive'
}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
soup

In [4]:
h3_tag=soup.find('h3', class_='c-finderProductCard_titleHeading')
game_name = h3_tag.find_all('span')[1].text.strip()
print(game_name) 

Disco Elysium: The Final Cut


In [5]:
release_date = soup.find('span',class_='u-text-uppercase').text.strip()
print(release_date)

Mar 30, 2021


In [ ]:
rated_span = soup.find('span',class_='u-text-capitalize')
game_rating = rated_span.next_sibling.strip()
print(game_rating)

In [7]:
game_description = soup.find('div',class_='c-finderProductCard_description').find('span').text.strip()
print(game_description)

Disco Elysium - The Final Cut is the definitive edition of the smash-hit RPG. Pursue your political dreams in new quests, meet and question more of the city's locals, and explore a whole extra area. Full voice-acting, controller support, and expanded language options also included. Get even more out of this award-winning open world. You're a detective with a unique skill system at your disposal and a whole city block to carve your path across. Interrogate unforgettable characters, crack murders, or take bribes. Become a hero or an absolute disaster of a human being.


In [8]:
game_score = soup.find('span', class_='u-flexbox-row').text.strip()
print(game_score)

97 Metascore


In [10]:
main_container = soup.find('div', class_='c-productListings')
# Find all game cards inside the main container
game_cards = main_container.find_all('div', class_='c-finderProductCard c-finderProductCard-game')

game_list = []
for game in game_cards:
    # Extract game name
    h3_tag = game.find('h3', class_='c-finderProductCard_titleHeading')
    game_name = h3_tag.find_all('span')[1].text.strip() if h3_tag else None

    # Extract release date
    release_date_tag = game.find('span', class_='u-text-uppercase')
    release_date = release_date_tag.text.strip() if release_date_tag else None

    # Extract rating (robust: get first non-empty text after the Rated span)
    rated_span = game.find('span', class_='u-text-capitalize')
    game_rating = None
    if rated_span:
        for sibling in rated_span.next_siblings:
            if isinstance(sibling, str) and sibling.strip():
                game_rating = sibling.strip()
                break

    # Extract description
    desc_tag = game.find('div', class_='c-finderProductCard_description')
    game_description = desc_tag.find('span').text.strip() if desc_tag and desc_tag.find('span') else None

    # Extract score
    game_score = game.find('span', class_='u-flexbox-row').text.strip() if game.find('span', class_='u-flexbox-row') else None

    # Append all extracted data to the list as a dictionary
    game_list.append({
        'Name': game_name,
        'Release Date': release_date,
        'Rating': game_rating,
        'Description': game_description,
        'Score': game_score
    })


# --- Comments ---
# - We first find the main container holding all games.
# - Then, we find each game card inside that container.
# - For each game, we extract the name, release date, rating, description, and score.
# - For the rating, we loop through next_siblings to get the first non-empty text (handles HTML quirks).
# - All data is stored in a list of dictionaries, one per game.

In [11]:
# now we will convert the list of dictionaries to a pandas dataframe
df = pd.DataFrame(game_list)
df

,Name,Release Date,Rating,Description,Score
0,Disco Elysium: The Final Cut,"Mar 30, 2021",M,Disco Elysium - The Final Cut is the definitiv...,97 Metascore
1,Half-Life 2,"Nov 16, 2004",M,[Metacritic's 2004 PC Game of the Year] By ta...,96 Metascore
2,Grand Theft Auto V,"Apr 13, 2015",M,"Los Santos is a vast, sun-soaked metropolis fu...",96 Metascore
3,Baldur's Gate 3,"Aug 3, 2023",M,"An ancient evil has returned to Baldur's Gate,...",96 Metascore
4,The Orange Box,"Oct 10, 2007",M,Games included in The Orange Box compilation: ...,96 Metascore
5,Half-Life,"Nov 19, 1998",M,Half-Life combines great storytelling in the t...,96 Metascore
6,BioShock,"Aug 21, 2007",M,"Going beyond ""run and gun corridors,"" ""monster...",96 Metascore
7,Baldur's Gate II: Shadows of Amn,"Sep 24, 2000",T,An epic continuation of the story that began i...,95 Metascore
8,Persona 5 Royal,"Oct 21, 2022",M,Prepare for an all-new RPG experience in Perso...,95 Metascore
9,Portal 2,"Apr 18, 2011",E10+,Valve is working on a full-length sequel to it...,95 Metascore


#### now we have a dataframe with all the data we want
#### we will now do this for all pages 

In [12]:
i=0
game_list=[]

while True:
    i+=1
    next_page_url = f'https://www.metacritic.com/browse/game/pc/all/all-time/metascore/?releaseYearMin=1958&releaseYearMax=2025&platform=pc&page={i}'
    response = requests.get(next_page_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    main_container = soup.find('div', class_='c-productListings')
    if not main_container:
        break  # Stop if there are no more pages
    game_cards = main_container.find_all('div', class_='c-finderProductCard c-finderProductCard-game')
    
    for game in game_cards:
        h3_tag = game.find('h3', class_='c-finderProductCard_titleHeading')
        game_name = h3_tag.find_all('span')[1].text.strip() if h3_tag else None

        release_date_tag = game.find('span', class_='u-text-uppercase')
        release_date = release_date_tag.text.strip() if release_date_tag else None

        rated_span = game.find('span', class_='u-text-capitalize')
        game_rating = None
        if rated_span:
            for sibling in rated_span.next_siblings:
                if isinstance(sibling, str) and sibling.strip():
                    game_rating = sibling.strip()
                    break

        desc_tag = game.find('div', class_='c-finderProductCard_description')
        game_description = desc_tag.find('span').text.strip() if desc_tag and desc_tag.find('span') else None

        game_score = game.find('span', class_='u-flexbox-row').text.strip() if game.find('span', class_='u-flexbox-row') else None

        game_list.append({
            'Name': game_name,
            'Release_Date': release_date,
            'Rating': game_rating,
            'Description': game_description,
            'Score': game_score
        })



In [13]:
df= pd.DataFrame(game_list)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6294 entries, 0 to 6293
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Name          6294 non-null   object
 1   Release_Date  6294 non-null   object
 2   Rating        3794 non-null   object
 3   Description   6294 non-null   object
 4   Score         6291 non-null   object
dtypes: object(5)
memory usage: 246.0+ KB


In [14]:
df.head(10)

,Name,Release_Date,Rating,Description,Score
0,Disco Elysium: The Final Cut,"Mar 30, 2021",M,Disco Elysium - The Final Cut is the definitiv...,97 Metascore
1,Half-Life 2,"Nov 16, 2004",M,[Metacritic's 2004 PC Game of the Year] By ta...,96 Metascore
2,Grand Theft Auto V,"Apr 13, 2015",M,"Los Santos is a vast, sun-soaked metropolis fu...",96 Metascore
3,Baldur's Gate 3,"Aug 3, 2023",M,"An ancient evil has returned to Baldur's Gate,...",96 Metascore
4,The Orange Box,"Oct 10, 2007",M,Games included in The Orange Box compilation: ...,96 Metascore
5,Half-Life,"Nov 19, 1998",M,Half-Life combines great storytelling in the t...,96 Metascore
6,BioShock,"Aug 21, 2007",M,"Going beyond ""run and gun corridors,"" ""monster...",96 Metascore
7,Baldur's Gate II: Shadows of Amn,"Sep 24, 2000",T,An epic continuation of the story that began i...,95 Metascore
8,Persona 5 Royal,"Oct 21, 2022",M,Prepare for an all-new RPG experience in Perso...,95 Metascore
9,Portal 2,"Apr 18, 2011",E10+,Valve is working on a full-length sequel to it...,95 Metascore


In [15]:
df.isnull().sum()

Name               0
Release_Date       0
Rating          2500
Description        0
Score              3
dtype: int64

In [20]:
# we have the full data now to convert it to csv
df.to_csv('../data/metacritic_Toppc_games.csv', index=False)

In [21]:
df

,Name,Release_Date,Rating,Description,Score
0,Disco Elysium: The Final Cut,"Mar 30, 2021",M,Disco Elysium - The Final Cut is the definitiv...,97 Metascore
1,Half-Life 2,"Nov 16, 2004",M,[Metacritic's 2004 PC Game of the Year] By ta...,96 Metascore
2,Grand Theft Auto V,"Apr 13, 2015",M,"Los Santos is a vast, sun-soaked metropolis fu...",96 Metascore
3,Baldur's Gate 3,"Aug 3, 2023",M,"An ancient evil has returned to Baldur's Gate,...",96 Metascore
4,The Orange Box,"Oct 10, 2007",M,Games included in The Orange Box compilation: ...,96 Metascore
...,...,...,...,...,...
6289,Alone in the Dark: Illumination,"Jun 11, 2015",None,A dire curse has shrouded the town of Lorwich ...,19 Metascore
6290,Ride to Hell: Retribution,"Jun 24, 2013",M,The game is set in the last years of the roari...,16 Metascore
6291,Mika and The Witch's Mountain,"Jan 22, 2025",None,"Meet Mika, an aspiring witch with a half-forme...",None
6292,Stormgate,"Aug 5, 2025",None,Earth didn't stand a chance. Our story begins ...,None


# we can use this csv file to make data base so we can query it later